Name: Mina Kumari Thapa  
Student ID: 3281874

In [ ]:
!pip install pyspark

In [ ]:
"""
PySpark Naive Bayes Spam Classification Pipeline
Dataset: emails.csv (bag-of-words matrix, 5172 emails x 3000 word-count features + Prediction label)
"""

import time
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, IDF, StringIndexer, Binarizer
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F

t0 = time.time()

In [ ]:
# STAGE 1: Spark Session Setup
# ---------------------------------------------------------------
spark = SparkSession.builder \
    .appName("SpamDetection_NaiveBayes") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.3


In [ ]:
# STAGE 2: Load Dataset
df = spark.read.csv('emails.csv', header=True, inferSchema=True)
df = df.withColumnRenamed("Email No.", "email_id")
print("Row count:", df.count())
print("Column count:", len(df.columns))
df.select("email_id", "the", "to", "gas", "meter", "Prediction").show(5)

Row count: 1039
Column count: 3002
+--------+---+---+---+-----+----------+
|email_id|the| to|gas|meter|Prediction|
+--------+---+---+---+-----+----------+
| Email 1|  0|  0|  0|    0|         0|
| Email 2|  8| 13|  1|    0|         0|
| Email 3|  0|  0|  2|    0|         0|
| Email 4|  0|  5|  0|    1|         0|
| Email 5|  7|  6|  2|    3|         0|
+--------+---+---+---+-----+----------+
only showing top 5 rows


In [ ]:

# Class balance
print("Class balance:")
df.groupBy("Prediction").count().show()

Class balance:
+----------+-----+
|Prediction|count|
+----------+-----+
|         1|  296|
|         0|  743|
+----------+-----+



In [ ]:
#  STAGE 3: Preprocessing
# ---------------------------------------------------------------
# NOTE: This CSV is already a pre-tokenised, pre-vectorised bag-of-words
# matrix (each of the 3000 columns is a vocabulary term, each cell is the
# raw term frequency / word count for that email). There is no raw email
# text column to run Tokenizer / StopWordsRemover / CountVectorizer on.
# We therefore treat the existing word-count columns as the CountVectorizer
# stage's output, and complete the TF-IDF pipeline by applying IDF on top
# of them (this is the standard second half of Spark's TF-IDF pipeline:
# HashingTF/CountVectorizer -> IDF).

feature_cols = [c for c in df.columns if c not in ("email_id", "Prediction")]
print("Number of vocabulary/feature columns:", len(feature_cols))

Number of vocabulary/feature columns: 3000


In [ ]:
# Label indexing (label is already binary 0/1, StringIndexer included to
# satisfy the standard ML pipeline stage and guarantee correct label typing)
label_indexer = StringIndexer(inputCol="Prediction", outputCol="label")

In [ ]:
# Assemble raw term-frequency columns into a single vector (== CountVectorizer output)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="tf_vector")

In [ ]:
# Apply IDF to convert raw term frequencies into TF-IDF weighted features
# (output column named "tfidf_features" since "features" is itself one of the
# 3000 vocabulary words in this dataset and would collide with a data column)
idf = IDF(inputCol="tf_vector", outputCol="tfidf_features")

In [ ]:
# STAGE 4: Train/Test Split
prep_df = label_indexer.fit(df).transform(df)
prep_df = assembler.transform(prep_df)
idf_model = idf.fit(prep_df)
prep_df = idf_model.transform(prep_df)

In [ ]:
#  Bernoulli NaiveBayes requires strictly binary (0/1) feature values, so we
# derive a separate "word present / absent" vector from the raw term-frequency
# vector using a Binarizer (threshold 0 => any word count > 0 becomes 1)
binarizer = Binarizer(inputCol="tf_vector", outputCol="binary_features", threshold=0.0)
prep_df = binarizer.transform(prep_df)

train_df, test_df = prep_df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())


Training rows: 869
Test rows: 170


In [ ]:
# STAGE 5: Train & Compare 3 Naive Bayes Variants
# ---------------------------------------------------------------
model_types = ["multinomial", "bernoulli", "complement"]
results = {}

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

In [ ]:
for mtype in model_types:
    print(f"\n=== Training NaiveBayes(modelType='{mtype}') ===")
    # Bernoulli NB models word presence/absence, so it needs the binary
    # feature vector; Multinomial and Complement work on the TF-IDF vector.
    feat_col = "binary_features" if mtype == "bernoulli" else "tfidf_features"
    nb = NaiveBayes(featuresCol=feat_col, labelCol="label", modelType=mtype, smoothing=1.0)
    model = nb.fit(train_df)
    preds = model.transform(test_df)

    acc = evaluator_acc.evaluate(preds)
    prec = evaluator_prec.evaluate(preds)
    rec = evaluator_rec.evaluate(preds)
    f1 = evaluator_f1.evaluate(preds)

    results[mtype] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")

    preds.select("email_id", "Prediction", "label", "prediction", "probability").show(5, truncate=60)


=== Training NaiveBayes(modelType='multinomial') ===
Accuracy:  0.9588
Precision: 0.9592
Recall:    0.9588
F1-score:  0.9590
+---------+----------+-----+----------+-----------------------------------------+
| email_id|Prediction|label|prediction|                              probability|
+---------+----------+-----+----------+-----------------------------------------+
|Email 100|       1.0|  1.0|       1.0|             [2.345677115258425E-113,1.0]|
|Email 104|       1.0|  1.0|       1.0|[9.975399309026393E-5,0.9999002460069099]|
|Email 106|       0.0|  0.0|       0.0|              [1.0,3.176949933710936E-48]|
|Email 110|       1.0|  1.0|       1.0|             [1.2627888865863382E-38,1.0]|
|Email 116|       1.0|  1.0|       1.0|                                [0.0,1.0]|
+---------+----------+-----+----------+-----------------------------------------+
only showing top 5 rows

=== Training NaiveBayes(modelType='bernoulli') ===
Accuracy:  0.8706
Precision: 0.8678
Recall:    0.8706
F1-sco

In [ ]:
#  STAGE 6: Comparison Summary
# ---------------------------------------------------------------
print("\n=== FINAL COMPARISON ===")
print(f"{'Model':<15}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}")
for mtype, m in results.items():
    print(f"{mtype:<15}{m['accuracy']:<12.4f}{m['precision']:<12.4f}{m['recall']:<12.4f}{m['f1']:<12.4f}")

best_model = max(results, key=lambda k: results[k]["f1"])
print(f"\nBest performing model (by F1-score): {best_model}")

print(f"\nTotal runtime: {time.time()-t0:.1f}s")
spark.stop()


=== FINAL COMPARISON ===
Model          Accuracy    Precision   Recall      F1          
multinomial    0.9588      0.9592      0.9588      0.9590      
bernoulli      0.8706      0.8678      0.8706      0.8686      
complement     0.9647      0.9655      0.9647      0.9649      

Best performing model (by F1-score): complement

Total runtime: 510.1s
